# FundusNet — Colab Pipeline

Setup, export ONNX from trained checkpoints, verify, and download.

**Runtime:** GPU (T4) required. Go to `Runtime > Change runtime type > T4 GPU`.

**Dataset:** Upload `retina_dataset.zip` to Google Drive before running.

## 1. Setup — Clone repo & install dependencies

In [ ]:
!git clone https://github.com/Mariakevin/FundusNet.git
%cd FundusNet

!pip install -q -r requirements.txt
!pip install -q timm onnxruntime

print("Setup complete!")

## 2. Mount Google Drive & extract dataset

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")

ZIP_PATH = "/content/drive/MyDrive/retina_dataset.zip"

if not os.path.exists(ZIP_PATH):
    print(f"ERROR: {ZIP_PATH} not found.")
    print("Upload retina_dataset.zip to Google Drive first!")
else:
    size_mb = os.path.getsize(ZIP_PATH) / 1e6
    print(f"Found {ZIP_PATH} ({size_mb:.0f} MB)")

    if os.path.exists("/content/retina_dataset") and len(os.listdir("/content/retina_dataset")) > 1:
        print("Dataset already extracted — skipping.")
    else:
        print("Extracting...")
        !unzip -q "{ZIP_PATH}" -d /content/
        print("Done!")

DATASET = "/content/retina_dataset"
print(f"\nDataset: {DATASET}")
total = 0
for cls in ["Healthy", "Cataract", "Glaucoma", "Retina Disease"]:
    cls_path = os.path.join(DATASET, cls)
    if os.path.exists(cls_path):
        count = len([f for f in os.listdir(cls_path) if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tiff"))])
        print(f"  {cls}: {count} images")
        total += count
    else:
        print(f"  {cls}: MISSING!")
print(f"  Total: {total} images")

## 3. Verify GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("ERROR: No GPU detected!")
    print("Go to Runtime > Change runtime type > T4 GPU")

## 4. Export Trained Models to ONNX

Run this **after** training completes. Exports each checkpoint to ONNX, verifies, and copies to Drive.

In [ ]:
import os
import subprocess

MODELS = ["swin", "efficientnet_v2", "convnext_v2", "deit3", "maxvit"]
DRIVE_DIR = "/content/drive/MyDrive"

print("=" * 60)
print("Export to ONNX + Save to Drive")
print("=" * 60)

exported = 0
skipped = 0
failed = 0

for model in MODELS:
    checkpoint = f"experiments/models/{model}_best.pth"
    onnx_local = f"models/{model}_retinopathy.onnx"
    onnx_drive = f"{DRIVE_DIR}/{model}_retinopathy.onnx"

    print(f"\n--- {model} ---")

    if os.path.exists(onnx_drive):
        print("  Already on Drive — skipping.")
        skipped += 1
        continue

    if not os.path.exists(checkpoint):
        print(f"  Checkpoint not found: {checkpoint}")
        print("  Train this model first, then re-run this cell.")
        failed += 1
        continue

    print("  Exporting...")
    export_cmd = f"python export_onnx.py --model {model} --checkpoint {checkpoint} --output {onnx_local} --verify"
    subprocess.run(export_cmd, shell=True)

    if os.path.exists(onnx_local):
        os.system(f"cp {onnx_local} {DRIVE_DIR}/")
        size_mb = os.path.getsize(onnx_local) / 1e6
        print(f"  Saved to Drive ({size_mb:.1f} MB)")
        exported += 1
    else:
        print("  ONNX export failed.")
        failed += 1

print(f"\n{'=' * 60}")
print(f"Done: {exported} exported, {skipped} skipped, {failed} failed")
print(f"{'=' * 60}")

## 5. Verify Exported ONNX Models

In [ ]:
import os

print("Models on Google Drive:")
print("-" * 50)
found = 0
for f in sorted(os.listdir("/content/drive/MyDrive")):
    if f.endswith(".onnx"):
        size_mb = os.path.getsize(f"/content/drive/MyDrive/{f}") / 1e6
        print(f"  {f}: {size_mb:.1f} MB")
        found += 1
if found == 0:
    print("  No ONNX models found on Drive.")

print("\nLocal models/ folder:")
print("-" * 50)
if os.path.exists("models"):
    found = 0
    for f in sorted(os.listdir("models")):
        if f.endswith(".onnx"):
            size_mb = os.path.getsize(f"models/{f}") / 1e6
            print(f"  {f}: {size_mb:.1f} MB")
            found += 1
    if found == 0:
        print("  No local ONNX models found.")
else:
    print("  No models/ folder.")

## 6. Quick Inference Test (ONNX Runtime)

In [ ]:
import time

import numpy as np
import onnxruntime as ort

categories = ["Healthy", "Cataract", "Glaucoma", "Retina Disease"]

print("Inference test:")
print("-" * 60)

for model in MODELS:
    path = f"models/{model}_retinopathy.onnx"
    if not os.path.exists(path):
        print(f"  {model}: NOT FOUND")
        continue

    sess = ort.InferenceSession(path, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
    dummy = np.random.randn(1, 3, 224, 224).astype(np.float32)

    for _ in range(3):
        sess.run(None, {"input": dummy})

    start = time.time()
    for _ in range(10):
        out = sess.run(None, {"input": dummy})
    ms = (time.time() - start) / 10 * 1000

    probs = np.exp(out[0][0]) / np.exp(out[0][0]).sum()
    pred = categories[np.argmax(probs)]
    conf = max(probs) * 100
    print(f"  {model}: {pred} ({conf:.1f}%) - {ms:.1f}ms")

print("-" * 60)

## 7. Download ONNX Files to Your Computer

Save them to `C:\Users\April\Desktop\retina_project\models\`.

In [ ]:
from google.colab import files

print("Downloading ONNX models...")
downloaded = 0
for f in sorted(os.listdir("/content/drive/MyDrive")):
    if f.endswith(".onnx"):
        print(f"  {f}...")
        files.download(f"/content/drive/MyDrive/{f}")
        downloaded += 1

if downloaded == 0:
    print("No ONNX models found on Drive.")
else:
    print(f"\nDownloaded {downloaded} models.")
    print("Save to C:\\Users\\April\\Desktop\\retina_project\\models\\")

## 8. Check Model Status

In [ ]:
import os

print("Model status:")
print("-" * 50)
for model in ["swin", "efficientnet_v2", "convnext_v2", "deit3", "maxvit"]:
    onnx_drive = f"/content/drive/MyDrive/{model}_retinopathy.onnx"
    checkpoint = f"experiments/models/{model}_best.pth"
    if os.path.exists(onnx_drive):
        size_mb = os.path.getsize(onnx_drive) / 1e6
        print(f"  {model}: DONE (ONNX on Drive, {size_mb:.1f} MB)")
    elif os.path.exists(checkpoint):
        print(f"  {model}: CHECKPOINT exists — run cell 4 to export")
    else:
        print(f"  {model}: NOT STARTED — train first")